In [27]:
import pandas as pd

In [28]:
df=pd.read_csv("Testing data.csv")

In [29]:
df.head()

,recency,history,used_discount,used_bogo,zip_code,is_referral,channel,offer,conversion
0,10,142.44,1,0,Surburban,0,Phone,Buy One Get One,0
1,6,329.08,1,1,Rural,1,Web,No Offer,0
2,7,180.65,0,1,Surburban,1,Web,Buy One Get One,0
3,9,675.83,1,0,Rural,1,Web,Discount,0
4,2,45.34,1,0,Urban,0,Web,Buy One Get One,0


In [30]:
df.shape

(64000, 9)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   recency        64000 non-null  int64  
 1   history        64000 non-null  float64
 2   used_discount  64000 non-null  int64  
 3   used_bogo      64000 non-null  int64  
 4   zip_code       64000 non-null  object 
 5   is_referral    64000 non-null  int64  
 6   channel        64000 non-null  object 
 7   offer          64000 non-null  object 
 8   conversion     64000 non-null  int64  
dtypes: float64(1), int64(5), object(3)
memory usage: 4.4+ MB


In [32]:
df.duplicated().sum()

np.int64(6603)

In [33]:
df["offer"].value_counts()

offer
Buy One Get One    21387
Discount           21307
No Offer           21306
Name: count, dtype: int64

In [34]:
df["conversion"].value_counts()

conversion
0    54606
1     9394
Name: count, dtype: int64

In [35]:
df[df.duplicated()]["offer"].value_counts()

offer
No Offer           2234
Buy One Get One    2228
Discount           2141
Name: count, dtype: int64

In [36]:
df_clean=df.drop_duplicates().copy()

In [37]:
df_clean["offer"].value_counts()

offer
Discount           19166
Buy One Get One    19159
No Offer           19072
Name: count, dtype: int64

In [38]:
print(df.shape,df_clean.shape)

(64000, 9) (57397, 9)


In [39]:
df_clean.groupby("offer")[["recency", "history"]].agg(["mean", "median"])

recency            history         
                     mean median        mean   median
offer                                                
Buy One Get One  5.651391    5.0  267.219681  185.060
Discount         5.688198    5.0  266.578219  182.735
No Offer         5.645973    5.0  265.483851  183.380

In [40]:
df_clean.groupby("offer")[["is_referral","used_discount","used_bogo"]].agg(["mean", "median"])

is_referral        used_discount        used_bogo       
                       mean median          mean median      mean median
offer                                                                   
Buy One Get One    0.503732    1.0      0.554256    1.0  0.556292    1.0
Discount           0.501774    1.0      0.557132    1.0  0.556663    1.0
No Offer           0.501206    1.0      0.557624    1.0  0.555055    1.0

In [41]:
df_clean.groupby("offer")["conversion"].agg(["sum","count","mean"])

,sum,count,mean
offer,,,
Buy One Get One,3069,19159,0.160186
Discount,3703,19166,0.193207
No Offer,2189,19072,0.114776


# DISCOUNT VS BOGO VS NO OFFER

In [42]:
from scipy.stats import chi2_contingency

In [43]:
table=pd.crosstab(df_clean["offer"],df_clean["conversion"])

In [44]:
table

conversion,0,1
offer,,
Buy One Get One,16090,3069
Discount,15463,3703
No Offer,16883,2189


In [45]:
chi2, p, dof, expected = chi2_contingency(table)


In [46]:
chi2_contingency(table)


Chi2ContingencyResult(statistic=np.float64(449.94011780493054), pvalue=np.float64(1.9803630984536353e-98), dof=2, expected_freq=array([[16167.8367162 ,  2991.1632838 ],
       [16173.74385421,  2992.25614579],
       [16094.41942959,  2977.58057041]]))

In [47]:
print(chi2, p, dof)

449.94011780493054 1.9803630984536353e-98 2


In [48]:
pd.DataFrame(expected, index=table.index, columns=table.columns).round(1)

conversion,0,1
offer,,
Buy One Get One,16167.8,2991.2
Discount,16173.7,2992.3
No Offer,16094.4,2977.6


In [49]:
%pip install statsmodels


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# DISCOUNT VS NO OFFER



In [50]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

In [51]:
converted = [3703, 2189]   
total = [19166, 19072] 

In [52]:
z, p = proportions_ztest(converted, total)


In [53]:
proportions_ztest(converted, total)

(np.float64(21.24018396268335), np.float64(4.062957983050675e-100))

In [54]:
print(z, p)

21.24018396268335 4.062957983050675e-100


In [55]:
low, high = confint_proportions_2indep(3703, 19166, 2189, 19072)
print(low, high)

0.07123869248259815 0.0856215349006843


# BOGO VS NO OFFER


In [56]:
converted2=[3069, 2189]
total2=[19159, 19072]


In [57]:
z2, p2=proportions_ztest(converted2, total2)

In [58]:
print(z2,p2)

12.89010662634312 5.117301079301091e-38


In [59]:
low2, high2 = confint_proportions_2indep(3069, 19159, 2189, 19072)

In [60]:
print (low2,high2)

0.03852228084481069 0.05229899349932258


# DISCOUNT VS BOGO

In [61]:
converted3=[3703, 3069]
total3=[19166, 19159]

In [62]:
z3, p3=proportions_ztest(converted3, total3)

In [63]:
print(z3,p3)

8.47429218721406 2.365131283310892e-17


In [64]:
low3, high3 = confint_proportions_2indep(3703, 19166, 3069, 19159)

In [65]:
print(low3,high3)

0.02538930641549309 0.04064984378950498


# Holm correction

In [66]:
from statsmodels.stats.multitest import multipletests

In [67]:
pvals = [p, p2, p3]

In [68]:
reject, p_adj, _, _ = multipletests(pvals, alpha=0.05, method="holm")

In [69]:
multipletests(pvals, alpha=0.05, method="holm")

(array([ True,  True,  True]),
 array([1.21888739e-99, 1.02346022e-37, 2.36513128e-17]),
 np.float64(0.016952427508441503),
 0.016666666666666666)

In [70]:
print(reject)
print(p_adj)

[ True  True  True]
[1.21888739e-99 1.02346022e-37 2.36513128e-17]
